# Lateral-Torsional Buckling — Shell Model (ASDShellQ4)

This notebook makes an I-beam buckle laterally-torsionally and checks the
critical moment against the classical closed form. The beam is modelled
with **ASDShellQ4** shell elements in the corotational formulation, loaded
by equal and opposite **strong-axis** end moments (uniform moment), with
a small lateral bow as the imperfection that triggers the buckling.

**What the notebook shows:**
- An I-section built from mid-surface plates that **share the web-flange
  junction lines**, so web and flanges are one connected section
- Transfinite quad mesh
- Mesh-level half-sine lateral imperfection (node perturbation)
- End moments as statically equivalent nodal forces (`g.loads.point.force_closest`)
- Fork supports declared node by node (`g.displacements.point`)
- A corotational openseespy deck, run past the bifurcation with
  displacement control
- The critical moment from a Southwell plot, **asserted** against the
  closed form, and the twist compared with the closed-form amplification
- Recording on the default route of each OpenSees build

### Classical LTB critical moment

For a doubly-symmetric I-section of span $L$ under uniform moment:

$$M_{cr} = \frac{\pi}{L}\sqrt{E I_y \, G J\left(1 + \frac{\pi^2 E C_w}{G J L^2}\right)}$$

where $I_y$ is the weak-axis second moment of area, $J$ the St. Venant
torsion constant and $C_w$ the warping constant.

**Boundary conditions it assumes** (fork supports at both ends): the lateral
displacement $u$ and the twist $\phi$ are prevented, and the lateral bending
rotation $u'$ and the warping $\phi'$ are free. It also assumes small
pre-buckling deflections (the in-plane bending is ignored) and no
distortion of the cross-section. The shell model reproduces the supports
with $u_x = u_y = 0$ at every end node and $u_z$ fixed at a single node,
so the end sections are free to warp and to rotate about both axes.

**The mutation.** Set `LTB_SHELL_MOMENT_AXIS=weak` before running and the
end moments act about the weak axis instead. A weak-axis moment does not
cause lateral-torsional buckling, so the asserts in section 8 must fail.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ── Section properties (mm, N, MPa) ─────────────────────
bf   = 127.0       # flange width
tf   = 8.5         # flange thickness
h    = 332.0       # clear web height (between the flanges' inner faces)
tw   = 5.8         # web thickness
L    = 4000.0      # span
d    = h + tf      # distance between the flange mid-surfaces

E    = 200_000.0   # Young's modulus (MPa)
nu   = 0.3
G    = E / (2 * (1 + nu))

# Properties of the mid-surface model: the web spans d, flange to flange.
I_y = (2 * tf * bf**3 + d * tw**3) / 12        # weak-axis I
I_x = 2 * (bf * tf**3 / 12 + bf * tf * (d / 2)**2) + tw * d**3 / 12   # strong-axis I
J   = (2 * bf * tf**3 + d * tw**3) / 3          # St. Venant torsion constant
C_w = tf * bf**3 * d**2 / 24                    # warping constant

warping = 1 + np.pi**2 * E * C_w / (G * J * L**2)
M_cr = (np.pi / L) * np.sqrt(E * I_y * G * J * warping)
print(f"I_y = {I_y:.4g} mm^4,  I_x = {I_x:.4g} mm^4")
print(f"J   = {J:.4g} mm^4,  C_w = {C_w:.4g} mm^6")
print(f"warping factor sqrt(1 + pi^2 E C_w / (G J L^2)) = {np.sqrt(warping):.3f}")
print(f"M_cr = {M_cr/1e6:.3f} kN·m")

# The moment axis: 'strong' is the LTB problem; 'weak' is the mutation
# that the asserts in section 8 must reject.
MOMENT_AXIS = os.environ.get("LTB_SHELL_MOMENT_AXIS", "strong")
if MOMENT_AXIS not in ("strong", "weak"):
    raise ValueError(
        f"LTB_SHELL_MOMENT_AXIS={MOMENT_AXIS!r}: use 'strong' or 'weak'")
print(f"End moments about the {MOMENT_AXIS} axis")

## 1. Geometry — five mid-surface plates sharing the junction lines

The web meets each flange along its centre line, so each flange is built
as two half-width plates on either side of that line, and the web uses the
same line. The plates then share the junction curves, and the mesh shares
their nodes: web and flanges act as one section.

In [ ]:
import sys
sys.path.insert(0, os.path.abspath('../../src'))

from apeGmsh import apeGmsh
import gmsh

g = apeGmsh(model_name='LTB_shell')
g.begin()
geo = g.model.geometry

y_tf = d / 2      # top flange mid-surface elevation
y_bf = -d / 2     # bottom flange mid-surface elevation

# Six points per end section: the flange tips (l, r) and the web-flange
# junctions (c) of the top (t) and bottom (b) flanges, at z = 0 and z = L.
pts = {}
for zk, z in (('0', 0.0), ('L', L)):
    for yk, y in (('t', y_tf), ('b', y_bf)):
        for xk, x in (('l', -bf / 2), ('c', 0.0), ('r', bf / 2)):
            pts[yk + xk + zk] = geo.add_point(x, y, z, sync=False)

lines = {}   # (start, end) -> line tag; each line is created once


def edge(a, b):
    """The line from point a to point b, shared by every plate that uses it."""
    if (b, a) in lines:
        return -lines[(b, a)]
    if (a, b) not in lines:
        lines[(a, b)] = geo.add_line(pts[a], pts[b], sync=False)
    return lines[(a, b)]


def add_plate(a, b):
    """The plate between the longitudinal lines through a and b."""
    loop = geo.add_curve_loop([edge(a + '0', b + '0'), edge(b + '0', b + 'L'),
                               edge(b + 'L', a + 'L'), edge(a + 'L', a + '0')],
                              sync=False)
    return geo.add_plane_surface(loop, sync=False)


s_tf = [add_plate('tl', 'tc'), add_plate('tc', 'tr')]   # top flange halves
s_bf = [add_plate('bl', 'bc'), add_plate('bc', 'br')]   # bottom flange halves
s_web = add_plate('bc', 'tc')                           # web, flange to flange
gmsh.model.occ.synchronize()

g.labels.add(2, s_tf, name='top_flange')
g.labels.add(2, s_bf, name='bot_flange')
g.labels.add(2, [s_web], name='web')
print(f"Surfaces: top_flange={s_tf}, bot_flange={s_bf}, web={s_web}")
print(f"Lines: {len(lines)} (the two junction lines are shared by three plates)")

## 2. Physical groups for BCs

In [ ]:
# The end sections are the lines with both ends at z = 0 (or z = L).
edges_z0 = [t for (a, b), t in lines.items() if a[-1] == b[-1] == '0']
edges_zL = [t for (a, b), t in lines.items() if a[-1] == b[-1] == 'L']

g.physical.add_curve(edges_z0, name='left_end')
g.physical.add_curve(edges_zL, name='right_end')
# One node carries the axial (uz) restraint: the top junction at z = 0.
g.physical.add_point([pts['tc0']], name='axial_pin')

print(f"left_end edges:  {edges_z0}")
print(f"right_end edges: {edges_zL}")

## 3. Transfinite mesh

In [ ]:
# Mesh control: element count along each edge
n_span   = 40   # elements along the span (z-direction)
n_flange = 6    # elements across the full flange width (x-direction)
n_web    = 16   # elements along the web height (y-direction)

mesh_struct = g.mesh.structured
for (a, b), tag in lines.items():
    if a[-1] != b[-1]:
        n = n_span            # longitudinal line
    elif a[0] == b[0]:
        n = n_flange // 2     # half-flange cross line
    else:
        n = n_web             # web cross line
    mesh_struct.set_transfinite_curve(tag, n + 1)

for s in [*s_tf, *s_bf, s_web]:
    mesh_struct.set_transfinite_surface(s)
    gmsh.model.mesh.setRecombine(2, s)

g.mesh.generation.generate(dim=2)

# The plates share their junction nodes: the count is one connected section.
info = gmsh.model.mesh.getNodes()
n_expected = (2 * (n_flange + 1) + (n_web + 1) - 2) * (n_span + 1)
print(f"Nodes: {len(info[0])} (connected section: {n_expected})")
assert len(info[0]) == n_expected, "web and flanges do not share their nodes"
for s_tag in [*s_tf, *s_bf, s_web]:
    etypes, etags, _ = gmsh.model.mesh.getElements(2, s_tag)
    n_elem = sum(len(t) for t in etags)
    print(f"  Surface {s_tag}: {n_elem} elements (type {list(etypes)})")

## 4. Apply half-sine imperfection to mesh nodes

Perturb all mesh nodes laterally (x-direction) by a half-sine
envelope along the span: $\Delta x = \delta_0 \sin(\pi z / L)$
where $\delta_0 = L / 1000$. Every node of a cross-section moves by the
same amount, so the imperfection is a lateral bow with no initial twist.

In [ ]:
imp_mag = L / 1000  # 4 mm imperfection at midspan

all_tags, all_coords, _ = gmsh.model.mesh.getNodes()
coords = np.array(all_coords).reshape(-1, 3)

# Half-sine perturbation in x, keyed by z
z_vals = coords[:, 2]
dx = imp_mag * np.sin(np.pi * z_vals / L)

coords[:, 0] += dx

# Write back perturbed coordinates
for i, tag in enumerate(all_tags):
    gmsh.model.mesh.setNode(int(tag), list(coords[i]), [])

print(f"Max lateral perturbation: {dx.max():.2f} mm at midspan")
print(f"Imperfection ratio: L/{int(L/imp_mag)}")

## 5. Loads, supports and FEM data extraction

In [ ]:
# ── Loads: equal-and-opposite end moments ─────────────────
# The end sections are curve groups (the plate edges at z = 0 and z = L),
# and a surface resultant load needs a surface target. So each end moment
# is declared as its statically equivalent nodal forces: the least-norm
# set f_i with  sum(f_i) = 0  and  sum((r_i - c) x f_i) = M  about the
# node centroid c of that end, f = A^T (A A^T)^-1 b, one load per node.
# For a moment about x this is f_z proportional to y: a linear bending
# stress over the end section.
def end_moment_forces(pg, moment):
    nodes = g.physical.get_nodes(1, g.physical.get_tag(1, pg))
    xyz = nodes['coords']
    assert len(set(nodes['tags'].tolist())) == len(xyz)
    arms = xyz - xyz.mean(axis=0)
    A = np.zeros((6, 3 * len(xyz)))
    for i, (rx, ry, rz) in enumerate(arms):
        A[0:3, 3*i:3*i + 3] = np.eye(3)
        A[3:6, 3*i:3*i + 3] = [[0, -rz, ry], [rz, 0, -rx], [-ry, rx, 0]]
    b = np.r_[0.0, 0.0, 0.0, moment]
    f = (A.T @ np.linalg.solve(A @ A.T, b)).reshape(-1, 3)
    # The resultant is the declared one: no net force, the moment about c.
    assert np.allclose(f.sum(axis=0), 0.0, atol=1e-9 * abs(M_cr) / d)
    assert np.allclose(np.cross(arms, f).sum(axis=0), moment,
                       rtol=1e-12, atol=1e-9 * abs(M_cr))
    return xyz, f

# Strong axis: the moment vector is along x, bending the beam in its
# vertical (y-z) plane. The weak-axis mutation turns it along y.
m_axis = {'strong': np.array([1.0, 0.0, 0.0]),
          'weak': np.array([0.0, 1.0, 0.0])}[MOMENT_AXIS]
with g.loads.case('moment'):
    for pg, moment in (('left_end', -M_cr * m_axis), ('right_end', M_cr * m_axis)):
        for xyz, f in zip(*end_moment_forces(pg, moment)):
            g.loads.point.force_closest(tuple(xyz), force=tuple(f), pg=pg)

# ── BCs: fork supports, declared node by node ─────────────
# ux and uy fixed at every end node: no lateral displacement, no twist and
# no vertical displacement of the end section. uz stays free at the end
# nodes, so the ends rotate about both axes and warp freely.
g.displacements.point(pg='left_end', dofs=[1, 1, 0])
g.displacements.point(pg='right_end', dofs=[1, 1, 0])
# uz at one node only removes the axial rigid-body mode. Each end's loads
# are self-equilibrated, so this support carries no force.
g.displacements.point(pg='axial_pin', dofs=[0, 0, 1])

# Extract FEM data
fem = g.mesh.queries.get_fem_data(dim=2)

print(f"Nodes: {len(fem.nodes.ids)}")
print(f"Loads: {fem.nodes.loads}")
print(f"SP:    {fem.nodes.sp}")

# Verify the total moment from the nodal end-moment forces at each end
id_to_idx = {int(nid): i for i, nid in enumerate(fem.nodes.ids)}
for pg in ('left_end', 'right_end'):
    end_ids = {int(n) for n in fem.nodes.select(pg=pg).ids}
    m_end = np.zeros(3)
    for rec in fem.nodes.loads:
        if rec.force_xyz is not None and int(rec.node_id) in end_ids:
            r = fem.nodes.coords[id_to_idx[rec.node_id]]
            m_end += np.cross(r, np.array(rec.force_xyz))
    print(f"Moment at {pg}: {m_end / 1e6} (kN·m)")

## 6. OpenSees model — native openseespy

ASDShellQ4 with the `-corotational` flag for geometric nonlinearity.
Supports are fixed directly on the end nodes (`g.displacements.point`), and
the moments are nodal force couples (`g.loads.point.force_closest`).

**Load control.** Near the critical moment the beam softens, so a run
under load control needs many Newton iterations per step and cannot pass
the bifurcation. The deck uses two phases instead:

1. **Load control** to $\lambda = M/M_{cr} = 0.6$ in three steps. The
   lateral displacement grows as $\lambda^2$ here, so displacement control
   cannot start from zero load (its first tangent is singular).
2. **Displacement control** on the lateral displacement of the shear
   centre at midspan, in 2 mm steps up to about 40 mm. The load factor
   follows the equilibrium path through the bifurcation.

The ASDShellQ4 corotational tangent converges linearly rather than
quadratically, so the deck uses `KrylovNewton`, which accelerates it to
about 10 iterations per step.

In [ ]:
import subprocess

venv_python = sys.executable   # this kernel's interpreter, which has openseespy

# ── Node data ─────────────────────────────────────────────
node_data = []
for nid, xyz in fem.nodes.select():
    node_data.append((int(nid), float(xyz[0]), float(xyz[1]), float(xyz[2])))

# ── Element data by label (for different shell thicknesses) ──
tf_ids, tf_conn = fem.elements.select(label='top_flange').result().resolve()
bf_ids, bf_conn = fem.elements.select(label='bot_flange').result().resolve()
web_ids, web_conn = fem.elements.select(label='web').result().resolve()

print(f"Elements: TF={len(tf_ids)}, BF={len(bf_ids)}, Web={len(web_ids)}")

# ── Midspan tracking nodes ────────────────────────────────
# The shear centre (web mid-height) and the two web-flange junctions.
# The imperfection moved them to x = delta_0 at midspan.
id_to_idx = {int(nid): i for i, nid in enumerate(fem.nodes.ids)}


def node_at(label, target):
    ids = sorted(int(n) for n in fem.nodes.select(label=label).ids)
    xyz = np.array([fem.nodes.coords[id_to_idx[n]] for n in ids])
    dist = np.linalg.norm(xyz - target, axis=1)
    assert dist.min() < 1e-6, f"no {label} node at {target}"
    return ids[int(np.argmin(dist))]


midspan_sc   = node_at('web', np.array([imp_mag, 0.0, L / 2]))
midspan_node = node_at('top_flange', np.array([imp_mag, y_tf, L / 2]))
midspan_bf   = node_at('bot_flange', np.array([imp_mag, y_bf, L / 2]))
# The junction nodes belong to the web too: the plates are connected.
assert {midspan_node, midspan_bf} <= {int(n) for n in fem.nodes.select(label='web').ids}

left_nodes  = sorted(fem.nodes.select(pg='left_end').ids)
right_nodes = sorted(fem.nodes.select(pg='right_end').ids)
print(f"Tracking: shear centre {midspan_sc}, top junction {midspan_node}, "
      f"bottom junction {midspan_bf}")
print(f"Left end: {len(left_nodes)} nodes, right end: {len(right_nodes)} nodes")

In [ ]:
# ── Build the openseespy script ──────────────────────────
n_load  = 3      # load-control steps to lambda = 0.6
lam_0   = 0.6
n_disp  = 19     # displacement-control steps
du      = 2.0    # mm of shear-centre lateral displacement per step
n_steps = n_load + n_disp

# Node data
node_lines = []
for nid, x, y, z in node_data:
    node_lines.append(f"ops.node({nid}, {x:.6f}, {y:.6f}, {z:.6f})")

# Element lines: section 1 for the flanges, section 2 for the web
elem_lines = []
for ids, conn, sec in ((tf_ids, tf_conn, 1), (bf_ids, bf_conn, 1),
                       (web_ids, web_conn, 2)):
    for e, c in zip(ids, conn):
        elem_lines.append(
            f"ops.element('ASDShellQ4', {int(e)}, {int(c[0])}, {int(c[1])}, "
            f"{int(c[2])}, {int(c[3])}, {sec}, '-corotational')")

# BCs from g.displacements.point records (direct fix on end nodes)
sp_fix = {}  # node_id -> set of dofs
for rec in fem.nodes.sp:
    sp_fix.setdefault(int(rec.node_id), set()).add(rec.dof)

fix_lines = []
for nid, dofs in sp_fix.items():
    mask = [1 if d_ in dofs else 0 for d_ in range(1, 7)]
    fix_lines.append(f"ops.fix({nid}, {mask[0]}, {mask[1]}, {mask[2]}, {mask[3]}, {mask[4]}, {mask[5]})")

# Load lines
load_lines = []
for rec in fem.nodes.loads:
    f = rec.force_xyz or (0, 0, 0)
    m = rec.moment_xyz or (0, 0, 0)
    load_lines.append(
        f"ops.load({int(rec.node_id)}, {f[0]:.10g}, {f[1]:.10g}, {f[2]:.10g}, "
        f"{m[0]:.10g}, {m[1]:.10g}, {m[2]:.10g})")

# Assemble script
script = f"""\
from apeGmsh.opensees.emitter.live import get_ops
ops = get_ops()   # the OpenSees module apeGmsh drives
import numpy as np
import json
import time

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

# ── Nodes ─────────────────────────────────────────────────
{chr(10).join(node_lines)}

# ── Sections ──────────────────────────────────────────────
ops.section('ElasticMembranePlateSection', 1, {E}, {nu}, {tf}, 0.0)   # flanges
ops.section('ElasticMembranePlateSection', 2, {E}, {nu}, {tw}, 0.0)   # web

# ── Elements (ASDShellQ4 -corotational) ───────────────────
{chr(10).join(elem_lines)}

# ── Boundary conditions (fork supports + one axial pin) ───
{chr(10).join(fix_lines)}

# ── Loading: end moments of M_cr, scaled by lambda ────────
ops.timeSeries('Linear', 1)
ops.pattern('Plain', 1, 1)
{chr(10).join(load_lines)}

# ── Analysis ──────────────────────────────────────────────
ops.constraints('Plain')
ops.numberer('RCM')
ops.system('UmfPack')
ops.test('NormDispIncr', 1e-5, 50)
ops.algorithm('KrylovNewton')
ops.integrator('LoadControl', {lam_0 / n_load})
ops.analysis('Static')

lambdas = []
u_sc = []    # shear-centre lateral displacement at midspan
ux_tf = []   # top junction lateral displacement
ux_bf = []   # bottom junction lateral displacement
iters = []
t_start = time.perf_counter()

for step in range({n_steps}):
    if step == {n_load}:
        # past the load-control phase: follow the shear centre laterally
        ops.integrator('DisplacementControl', {midspan_sc}, 1, {du})
    ok = ops.analyze(1)
    if ok != 0:
        print(f"Analysis failed at step {{step}}, lambda={{ops.getTime():.4f}}")
        break
    lam = ops.getTime()
    u_top = ops.nodeDisp({midspan_node}, 1)
    u_bot = ops.nodeDisp({midspan_bf}, 1)
    lambdas.append(lam)
    u_sc.append(ops.nodeDisp({midspan_sc}, 1))
    ux_tf.append(u_top)
    ux_bf.append(u_bot)
    iters.append(ops.testIter())
    twist = np.degrees(np.arcsin((u_top - u_bot) / {d}))
    print(f"  step {{step:3d}}: lambda={{lam:.4f}}, u_sc={{u_sc[-1]:7.3f}} mm, "
          f"twist={{twist:7.3f}} deg, {{iters[-1]}} iterations")

print(f"\\nCompleted {{len(lambdas)}} steps, final lambda = {{lambdas[-1]:.4f}}, "
      f"{{sum(iters)}} Newton iterations in {{time.perf_counter() - t_start:.0f}} s")

results = {{
    'lambdas': lambdas,
    'u_sc': u_sc,
    'ux_tf': ux_tf,
    'ux_bf': ux_bf,
    'iters': iters,
    'M_cr': {M_cr},
}}
with open('ltb_shell_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f)
print("Results saved")
"""

script_path = 'ltb_shell_analysis.py'
with open(script_path, 'w', encoding='utf-8') as f:
    f.write(script)
print(f"Script: {len(script.splitlines())} lines, {len(elem_lines)} elements")

## 7. Run analysis

In [ ]:
# 22 steps at about 10 KrylovNewton iterations each take well under a
# minute on a desktop; the timeout only guards against a hung process.
import time
t0 = time.perf_counter()
result = subprocess.run(
    [venv_python, script_path],
    capture_output=True, text=True, timeout=1800
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
print(f"Subprocess wall time: {time.perf_counter() - t0:.0f} s")

## 8. Results & comparison to classical M_cr

**Second-order theory.** With fork supports, a uniform moment $M$ and an
initial lateral bow $u_0 \sin(\pi z/L)$ (no initial twist), the additional
lateral displacement $u$ of the shear centre and the twist $\phi$ at
midspan are, with $m = M/M_{cr}$:

$$u = u_0\,\frac{m^2}{1 - m^2}, \qquad
\phi = \frac{M\,u_0}{GJ + \pi^2 E C_w / L^2}\;\frac{1}{1 - m^2}$$

Both grow without bound as $M \to M_{cr}$: the bifurcation.

**Southwell plot.** The first relation rearranges to a straight line in $u$:

$$\frac{u}{\lambda^2} = \frac{u}{\lambda_{cr}^2} + \frac{u_0}{\lambda_{cr}^2}$$

where $\lambda = M / M_{cr}^{\text{closed form}}$ is the deck's load factor.
The slope of the computed $u/\lambda^2$ against $u$ gives $\lambda_{cr}$, so
$M_{cr}^{FE} = \lambda_{cr} M_{cr}$, without assuming the closed form; the
intercept over the slope gives back $u_0$, which must be the 4 mm imposed.
The fit uses the points with $0.5 \le u \le 2.5\,u_0$ mm (twist below about
3.5°), where the small-twist theory holds. Beyond it, the corotational
shell stiffens in the post-buckling range and the line bends upward.

**Tolerances, asserted below.**
- $M_{cr}^{FE}$ within **5 %** of the closed form. The shell model sits a
  little above it: the in-plane pre-buckling deflection, which the closed
  form ignores, raises $M_{cr}$ by about $1/\sqrt{1 - I_y/I_x} \approx 1.8\,\%$.
- The recovered $u_0$ within **10 %** of the imposed imperfection.
- The twist within **5 %** of the closed-form $\phi$ in the fit window, and
  growing past the bifurcation: at the last step it exceeds 5°, and its
  rate $d\phi/d\lambda$ is more than 10 times that of the first step.

A weak-axis moment (the mutation) bends the flanges in their own plane and
does not twist the section, so these asserts fail.

In [ ]:
import json

with open('ltb_shell_results.json') as f:
    res = json.load(f)

lam = np.array(res['lambdas'])
u_sc = np.array(res['u_sc'])
ux_top = np.array(res['ux_tf'])
ux_bot = np.array(res['ux_bf'])
M_applied = lam * M_cr           # loads were scaled to M_cr

# Twist from the junction displacements (the web stays straight)
twist = np.arcsin((ux_top - ux_bot) / d)       # rad
u_0 = imp_mag

# ── Southwell plot: u / lambda^2 against u ───────────────
window = (u_sc >= 0.5) & (u_sc <= 2.5 * u_0)
if window.sum() >= 4:
    slope, intercept = np.polyfit(u_sc[window], u_sc[window] / lam[window]**2, 1)
else:
    slope = intercept = np.nan   # no buckling path to fit
lam_cr = 1 / np.sqrt(slope) if slope > 0 else np.nan
M_cr_fe = lam_cr * M_cr
u_0_fe = intercept / slope

# ── Closed-form second-order response ────────────────────
r_t = G * J + np.pi**2 * E * C_w / L**2        # torsional stiffness term
M_range = np.linspace(0, 0.995, 400) * M_cr
m_r = M_range / M_cr
u_theory = u_0 * m_r**2 / (1 - m_r**2)
phi_theory = M_range * u_0 / r_t / (1 - m_r**2)
phi_at_steps = M_applied * u_0 / r_t / (1 - lam**2)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(17, 5))

ax1.plot(u_sc, M_applied / 1e6, 'b-o', ms=3, label='Shell: shear centre')
ax1.plot(u_theory, M_range / 1e6, 'r--', label='Second-order theory')
ax1.axhline(M_cr / 1e6, color='gray', ls=':', label=f'$M_{{cr}}$ = {M_cr/1e6:.2f} kN·m')
ax1.set_xlabel('Lateral displacement u (mm)')
ax1.set_ylabel('Applied moment M (kN·m)')
ax1.set_title('M vs lateral displacement')
ax1.set_xlim(0, max(u_sc.max(), 1.0) * 1.05)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(np.degrees(np.abs(twist)), M_applied / 1e6, 'g-o', ms=3, label='Shell twist')
ax2.plot(np.degrees(phi_theory), M_range / 1e6, 'r--', label='Second-order theory')
ax2.axhline(M_cr / 1e6, color='gray', ls=':', label=f'$M_{{cr}}$ = {M_cr/1e6:.2f} kN·m')
ax2.set_xlabel('Twist |φ| at midspan (deg)')
ax2.set_ylabel('Applied moment M (kN·m)')
ax2.set_title('M vs twist')
ax2.set_xlim(0, max(np.degrees(np.abs(twist)).max(), 1.0) * 1.05)
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3.plot(u_sc, u_sc / lam**2, 'k-o', ms=3, label='Shell')
ax3.plot(u_sc[window], u_sc[window] / lam[window]**2, 'bo', ms=6, label='Fit window')
if np.isfinite(slope):
    ax3.plot(u_sc, slope * u_sc + intercept, 'r--',
             label=f'Fit: $\\lambda_{{cr}}$ = {lam_cr:.3f}')
ax3.set_xlabel('u (mm)')
ax3.set_ylabel('u / λ² (mm)')
ax3.set_title('Southwell plot')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Report: twist grows at the bifurcation ───────────────
# The closed-form twist is small-twist theory: it is listed in the fit
# window only, where it applies.
print(f"{'lambda':>8} {'M (kN·m)':>9} {'u_sc (mm)':>10} {'twist (deg)':>12} {'theory (deg)':>13}")
for i in range(len(lam)):
    theory = f"{np.degrees(phi_at_steps[i]):13.3f}" if window[i] else f"{'-':>13}"
    print(f"{lam[i]:8.4f} {M_applied[i]/1e6:9.2f} {u_sc[i]:10.3f} "
          f"{np.degrees(abs(twist[i])):12.3f} {theory}")

pre_buckling = 1 / np.sqrt(1 - I_y / I_x)
print(f"\nClassical M_cr      = {M_cr/1e6:.3f} kN·m (fork supports, with warping)")
print(f"Southwell M_cr (FE) = {M_cr_fe/1e6:.3f} kN·m  (lambda_cr = {lam_cr:.4f}, "
      f"{window.sum()} points)")
print(f"Pre-buckling estimate 1/sqrt(1 - I_y/I_x) = {pre_buckling:.4f}")
print(f"Recovered imperfection u_0 = {u_0_fe:.3f} mm (imposed {u_0:.3f} mm)")
print(f"Twist at the last step: {np.degrees(abs(twist[-1])):.2f} deg at lambda = {lam[-1]:.4f}")

# ── The asserts: the oracle of this notebook ─────────────
assert abs(lam_cr - 1) < 0.05, (
    f"Southwell M_cr = {M_cr_fe/1e6:.3f} kN·m is not within 5 % of the "
    f"closed-form {M_cr/1e6:.3f} kN·m")
assert abs(u_0_fe / u_0 - 1) < 0.10, (
    f"the Southwell intercept gives u_0 = {u_0_fe:.3f} mm, not the imposed {u_0} mm")
assert np.allclose(np.abs(twist[window]), phi_at_steps[window], rtol=0.05), (
    "the twist departs from the closed-form second-order twist")
rate = np.abs(np.diff(np.r_[0.0, twist])) / np.diff(np.r_[0.0, lam])
assert np.degrees(abs(twist[-1])) > 5 and rate[-1] > 10 * rate[0], (
    "the twist does not grow at the bifurcation")
assert len(lam) == n_steps, f"the run stopped after {len(lam)} of {n_steps} steps"
print("\nLTB checks pass: M_cr, imperfection, twist amplitude and twist growth")

## 9. Record results

Section 7 ran the deck in a subprocess. The cell below runs the same deck
in this kernel and records it. The deck binds ``ops = get_ops()``, and the
cell picks the route by ``get_backend_info().kind``:

- **Ladruno fork:** ``ResolvedRecorderSpec.emit_default`` records the load
  steps with the fork's own recorder (a ``.ladruno`` file), read back with
  ``Results.from_ladruno``.
- **Stock openseespy:** the cell samples the live domain after every
  converged step and writes a native results file with ``NativeWriter``,
  read back with ``Results.from_fem``. That file holds **displacements
  only**: reaction forces are recorded on the fork only. ``emit_default``
  cannot record this deck on a stock build without MPCO: its ``.out``
  route needs a bridge model archive that declares ``ndm``, and this deck
  was built by hand.

The deck tags nodes and elements by their FEM ids, so the FEMData snapshot
is the model. The cell checks the recorded last step against the live
domain, and the recorded midspan history against the subprocess run of
section 7. ``Results.viewer`` honours ``APEGMSH_SKIP_VIEWER``, so this
notebook is safe to run under nbconvert / CI without changes.

In [ ]:
# --- EOS-WIRING-V2 ---
# Record the deck's load steps in-process: the fork's own recorder on the
# Ladruno fork, a native file of displacements sampled from the live
# domain on stock openseespy.
import contextlib
from apeGmsh import Results, workdir
from apeGmsh.opensees.emitter.live import get_backend_info, get_ops
from apeGmsh.results.spec import ResolvedRecorderRecord, ResolvedRecorderSpec
from apeGmsh.results.writers import NativeWriter

OUT = workdir()

# The deck in two parts: the model and its analysis chain, then the load
# steps (without the JSON dump, which belongs to the subprocess run).
# The steps call _record_step() after every converged ops.analyze.
deck_model, _sep, _rest = script.partition("lambdas = []\n")
deck_steps = _sep + _rest.partition("\nresults = {")[0]
_hook = "    ok = ops.analyze(1)\n"
assert deck_steps.count("ops.analyze(") == deck_steps.count(_hook) == 1
deck_steps = deck_steps.replace(
    _hook, _hook + "    if ok == 0:\n        _record_step()\n")
deck_ns = {}
exec(deck_model, deck_ns)
ops = deck_ns["ops"]

_axes = ("x", "y", "z")
_built = set(int(n) for n in ops.getNodeTags())
_node_ids = np.asarray(
    [int(n) for n in fem.nodes.ids if int(n) in _built], dtype=np.int64)
# The deck drives the module get_ops() binds; get_backend_info() names it.
assert ops is get_ops(), "bind the deck with ops = get_ops()"
_backend = get_backend_info()
_fork = _backend.kind == "fork"
print(f"OpenSees build: {_backend.kind} ({_backend.source})")

if _fork:
    spec = ResolvedRecorderSpec(
        fem_snapshot_id=fem.snapshot_id,
        records=tuple(
            ResolvedRecorderRecord(
                category="nodes", name=kind,
                components=tuple(f"{kind}_{a}" for a in _axes),
                dt=None, n_steps=None, node_ids=_node_ids,
            )
            for kind in ("displacement", "reaction_force")
        ),
    )
    # emit_default needs a model archive on disk; the deck tags nodes and
    # elements by FEM id, so the FEMData archive is that model.
    model_h5 = OUT / "model.h5"
    fem.to_h5(str(model_h5))
    _recording = spec.emit_default(OUT / "run", model_h5=model_h5, ops=ops,
                                   overwrite=True)
else:
    _recording = contextlib.nullcontext()

_steps = []   # stock: (time, displacement) after every converged step


def _record_step():
    if not _fork:
        _steps.append((ops.getTime(), np.array(
            [ops.nodeDisp(int(n))[:3] for n in _node_ids])))


deck_ns["_record_step"] = _record_step
with _recording as run:
    exec(deck_steps, deck_ns)

if _fork:
    results_path = run.path
    results = Results.from_ladruno(str(results_path), model_h5=str(model_h5))
else:
    results_path = OUT / "run_native.h5"
    if results_path.exists():
        results_path.unlink()
    _u = np.stack([u for _, u in _steps])          # (T, N, 3)
    with NativeWriter(results_path) as _nw:
        _nw.open(fem=fem, source_type="native", opensees_backend=_backend.kind)
        _sid = _nw.begin_stage(name="analysis", kind="static",
                               time=np.array([t for t, _ in _steps]))
        _nw.write_nodes(_sid, "partition_0", node_ids=_node_ids, components={
            f"displacement_{a}": _u[:, :, k] for k, a in enumerate(_axes)})
        _nw.end_stage()
    results = Results.from_fem(fem, results_path, kind="native", cache_root=OUT)
print(f"recorded -> {results_path}")

# The recorded last step is the live domain's displacement field.
_slabs = [results.nodes.get(ids=_node_ids, component=f"displacement_{a}")
          for a in _axes]
assert sorted(int(n) for n in _slabs[0].node_ids) == sorted(_node_ids.tolist())
_rec = np.column_stack([s.values[-1] for s in _slabs])
_live = np.array([ops.nodeDisp(int(n))[:3] for n in _slabs[0].node_ids])
assert np.allclose(_rec, _live, rtol=1e-6, atol=1e-9 * float(np.abs(_live).max()))
print(f"recorded last step matches the live domain at {len(_node_ids)} nodes")

# The recorded midspan history is the subprocess run's (section 7).
_tf = results.nodes.get(ids=[midspan_node], component="displacement_x").values[:, 0]
assert np.allclose(_tf, res['ux_tf'], rtol=1e-6, atol=1e-9), (
    "the in-process run disagrees with the subprocess run")
print(f"recorded midspan ux over {len(_tf)} steps matches the subprocess run")


In [ ]:
# Open the recorded run in the apeGmsh ResultsViewer (subprocess, non-blocking).
# Set APEGMSH_SKIP_VIEWER=1 in the environment to skip the GUI under nbconvert / CI.
results.viewer(blocking=False)


In [ ]:
g.end()
print('Done')